# PyWry TradingView Lightweight Charts Demo

This notebook demonstrates the minimum viable code required to render a TradingView chart using PyWry. It covers two concepts - static time series & UDF Adapter. 

In [ ]:
import random
import time


def generate_ohlcv(days=250, start_price=150.0, seed=42):
    """Generate synthetic daily OHLCV bars."""
    random.seed(seed)
    bars = []
    price = start_price
    t = int(time.time()) - days * 86400

    for _ in range(days):
        change = random.gauss(0, 2)
        o = round(price, 2)
        c = round(price + change, 2)
        h = round(max(o, c) + abs(random.gauss(0, 0.5)), 2)
        l = round(min(o, c) - abs(random.gauss(0, 0.5)), 2)
        v = random.randint(500_000, 5_000_000)
        bars.append({"time": t, "open": o, "high": h, "low": l, "close": c, "volume": v})
        price = c
        t += 86400

    return bars


data = generate_ohlcv()
print(f"Generated {len(data)} bars")

In [ ]:
from pywry import PyWry

app = PyWry()
widget = app.show_tvchart(data)

## With Toolbars

Use `build_tvchart_toolbars()` to get default chart controls (chart type selector,
drawing tools, time range presets).

In [ ]:
from pywry.tvchart import build_tvchart_toolbars

widget2 = app.show_tvchart(
    data,
    symbol="Mock Symbol",
    height=600,
    toolbars=build_tvchart_toolbars(),
)

## UDF Adapter

In [ ]:
from pywry import PyWry
from pywry.tvchart.udf import UDFAdapter


# BitMEX UDF Adapter for TradingView
UDF_URL = "https://www.bitmex.com/api/udf"

app = PyWry()

udf = UDFAdapter(UDF_URL, poll_interval=60)

handle = udf.connect(
    app,
    symbol="XBTUSD",
    resolution="D",
)


## Yield Curve Chart

`chart_kind="yield-curve"` swaps the time X axis for a linearly-spaced tenor axis, rendering each point at a fixed month-unit stride.  Use for treasury / SOFR / swap / credit curves and any "value vs tenor" term structure view.  Each datapoint is `{time: <months>, value: <yield_pct>}`.


In [ ]:
# US Treasury yield curve — representative snapshot.
# Each row maps a tenor (months) to the par yield at that tenor.
treasury_curve = [
    {"time": 1,   "value": 4.38},   # 1M
    {"time": 3,   "value": 4.30},   # 3M
    {"time": 6,   "value": 4.12},   # 6M
    {"time": 12,  "value": 3.95},   # 1Y
    {"time": 24,  "value": 3.80},   # 2Y
    {"time": 36,  "value": 3.86},   # 3Y
    {"time": 60,  "value": 4.05},   # 5Y
    {"time": 84,  "value": 4.22},   # 7Y
    {"time": 120, "value": 4.35},   # 10Y
    {"time": 240, "value": 4.62},   # 20Y
    {"time": 360, "value": 4.58},   # 30Y
]

app.show_tvchart(
    data=treasury_curve,
    title="US Treasury Yield Curve",
    height=620,
    chart_kind="yield-curve",
    yield_curve={
        "baseResolution": 1,      # one month per step
        "minimumTimeRange": 360,  # 30 years visible by default
        "startTimeRange": 0,
    },
    series_options={
        "seriesType": "Line",
        "color": "#4c87ff",
        "lineWidth": 2,
    },
)


## Options Payoff Chart

`chart_kind="price"` swaps the time X axis for a numeric price / strike axis (via `createOptionsChart`).  Use for option payoff profiles, IV smile/skew, volume-by-strike, market-profile histograms, order-book depth — anything indexed by price rather than by time.  Each datapoint is `{time: <price>, value: <...>}`.


In [ ]:
# Long-call payoff at expiry:  max(S - K, 0) - premium
def call_payoff(strike: float, premium: float) -> list[dict[str, float]]:
    return [
        {
            "time": round(s, 2),
            "value": round(max(s - strike, 0.0) - premium, 2),
        }
        for s in (round(strike - 50.0 + i, 2) for i in range(151))
    ]


strike, premium = 275.0, 5.20
payoff = call_payoff(strike, premium)

app.show_tvchart(
    data=payoff,
    title=f"Long Call — Strike ${strike:.0f}, Premium ${premium:.2f}",
    height=620,
    chart_kind="price",
    series_options={
        "seriesType": "Line",
        "color": "#26a69a",
        "lineWidth": 2,
    },
)
